In [1]:
# Transformers and Hugging Face Libraries
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# Hugging Face Datasets
from datasets import load_dataset, Dataset

# Utilities for Tokenization and Evaluation
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from itertools import chain
import numpy as np

from itertools import chain
import random

## 1.1: Load the GPT-2 Model

In [2]:
# Load the GPT-2 model pre-trained for sequence classification
model_name = "gpt2"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set the padding token to avoid errors
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded GPT-2 model and tokenizer for sequence classification.")

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded GPT-2 model and tokenizer for sequence classification.


## 1.2: Load and Preprocess the Dataset

In [3]:
# Load the IMDB dataset
dataset = load_dataset("imdb")

# Define subset size (200 samples total: 100 negative + 100 positive)
subset_size = 200

# Ensure balanced subsets for training and testing
negative_samples_train = [sample for sample in dataset["train"] if sample["label"] == 0][:subset_size // 2]
positive_samples_train = [sample for sample in dataset["train"] if sample["label"] == 1][:subset_size // 2]
balanced_train_data = list(chain(negative_samples_train, positive_samples_train))

negative_samples_test = [sample for sample in dataset["test"] if sample["label"] == 0][:subset_size // 2]
positive_samples_test = [sample for sample in dataset["test"] if sample["label"] == 1][:subset_size // 2]
balanced_test_data = list(chain(negative_samples_test, positive_samples_test))

# Shuffle the data to avoid label order bias

random.shuffle(balanced_train_data)
random.shuffle(balanced_test_data)

# Convert to Dataset objects
train_data = Dataset.from_dict({key: [sample[key] for sample in balanced_train_data] for key in balanced_train_data[0]})
test_data = Dataset.from_dict({key: [sample[key] for sample in balanced_test_data] for key in balanced_test_data[0]})

# Check label distribution
print(f"Train labels: {train_data['label'].count(0)} {train_data['label'].count(1)}")
print(f"Test labels: {test_data['label'].count(0)} {test_data['label'].count(1)}")


# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Apply tokenization
tokenized_train = train_data.map(tokenize_function, batched=True, desc="Tokenizing Train Data")
tokenized_test = test_data.map(tokenize_function, batched=True, desc="Tokenizing Test Data")

# Set format for PyTorch
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("Balanced dataset loaded and tokenized.")

Train labels: 100 100
Test labels: 100 100


Tokenizing Train Data:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing Test Data:   0%|          | 0/200 [00:00<?, ? examples/s]

Balanced dataset loaded and tokenized.


## 1.3: Evaluate the Pretrained Model

In [7]:
# Define a function to compute evaluation metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Set up TrainingArguments for evaluation
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_eval_batch_size=8,
    logging_dir="./logs"
)

# Create a Trainer object for evaluation
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Evaluate the pretrained model
evaluation_results = trainer.evaluate()
print(f"Evaluation Results: {evaluation_results}")


C:\Users\toto1\anaconda3\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\toto1\AppData\Local\Temp\ipykernel_11724\1728061350.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation Results: {'eval_loss': 1.6724305152893066, 'eval_model_preparation_time': 0.0312, 'eval_accuracy': 0.49, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_runtime': 272.213, 'eval_samples_per_second': 0.735, 'eval_steps_per_second': 0.092}


# 2. Perform Lightweight Fine-Tuning

## Step 2.1: Create a PEFT Model

In [9]:
# Create a LoRA configuration
peft_config = LoraConfig(
    r=4,  # Reduced rank to prevent overfitting
    lora_alpha=16,  # Adjusted scaling
    target_modules=["c_attn"],  # GPT-2 attention layer
    lora_dropout=0.2,  # Increased dropout to prevent overfitting
    bias="none",
    task_type="SEQ_CLS"  # Correct task type for sequence classification
)

# Wrap the GPT-2 model with LoRA
peft_model = get_peft_model(model, peft_config)
print("LoRA-enabled model created.")

C:\Users\toto1\anaconda3\Lib\site-packages\peft\tuners\lora\layer.py:1264: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


LoRA-enabled model created.


## Step 2.2: Train the PEFT Model

In [11]:
# Update training arguments for fine-tuning
training_args = TrainingArguments(
    output_dir="./PEFT Model",
    num_train_epochs=1,  # Increased epochs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,  # Reduced accumulation steps
    learning_rate=2e-5,  # Lowered learning rate
    evaluation_strategy="epoch",  # Evaluate after each epoch
    save_strategy="epoch",
    fp16=True,  # Mixed precision for faster training
    logging_dir="./logs",
    save_total_limit=2,
    remove_unused_columns=False
)

# Create a Trainer for fine-tuning
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Fine-tune the model
trainer.train()

C:\Users\toto1\anaconda3\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\toto1\AppData\Local\Temp\ipykernel_11724\198416952.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
0,No log,1.591112,0.490000,0.000000,0.000000,0.000000


C:\Users\toto1\anaconda3\Lib\site-packages\peft\utils\other.py:716: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 3cb0afff-396a-4d66-b3ac-0d46f59d9ca2)') - silently ignoring the lookup for the file config.json in gpt2.
  warnings.warn(
C:\Users\toto1\anaconda3\Lib\site-packages\peft\utils\save_and_load.py:246: UserWarning: Could not find a config file in gpt2 - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=12, training_loss=1.860737959543864, metrics={'train_runtime': 1930.358, 'train_samples_per_second': 0.104, 'train_steps_per_second': 0.006, 'total_flos': 50256855171072.0, 'train_loss': 1.860737959543864, 'epoch': 0.96})

## Step 2.3: Save the PEFT Model

In [13]:
# Save the fine-tuned LoRA model
save_directory = "./fine_tuned_peft_model"
peft_model.save_pretrained(save_directory)
print(f"Model saved to {save_directory}.")

Model saved to ./fine_tuned_peft_model.


### Step 3.1: Load the Saved PEFT Model

In [23]:
# Load the fine-tuned model
# Load the fine-tuned model
fine_tuned_model = AutoModelForSequenceClassification.from_pretrained(save_directory, num_labels=2)

# Ensure the fine-tuned model recognizes the padding token
fine_tuned_model.config.pad_token_id = tokenizer.pad_token_id

print("Fine-tuned model loaded with padding token set.")


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fine-tuned model loaded with padding token set.


In [25]:
# Retry evaluation with the fixed padding token
trainer_fine_tuned = Trainer(
    model=fine_tuned_model,
    args=training_args,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Evaluate the fine-tuned model
fine_tuned_results = trainer_fine_tuned.evaluate()

# Compare metrics
print("\nComparison of Model Performance Metrics:")
print("Original Model Results:")
for metric, value in evaluation_results.items():
    print(f"  {metric}: {value:.4f}")

print("\nFine-Tuned Model Results:")
for metric, value in fine_tuned_results.items():
    print(f"  {metric}: {value:.4f}")

# Highlight improvement or regression
print("\nMetric Comparison (Improvement/Regression):")
for metric in evaluation_results.keys():
    if metric in fine_tuned_results:
        change = fine_tuned_results[metric] - evaluation_results[metric]
        print(f"  {metric}: {'+' if change >= 0 else ''}{change:.4f}")


C:\Users\toto1\AppData\Local\Temp\ipykernel_11724\4233928729.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_fine_tuned = Trainer(



Comparison of Model Performance Metrics:
Original Model Results:
  eval_loss: 1.6724
  eval_model_preparation_time: 0.0312
  eval_accuracy: 0.4900
  eval_precision: 0.0000
  eval_recall: 0.0000
  eval_f1: 0.0000
  eval_runtime: 272.2130
  eval_samples_per_second: 0.7350
  eval_steps_per_second: 0.0920

Fine-Tuned Model Results:
  eval_loss: 1.5911
  eval_model_preparation_time: 0.0313
  eval_accuracy: 0.4900
  eval_precision: 0.0000
  eval_recall: 0.0000
  eval_f1: 0.0000
  eval_runtime: 279.2237
  eval_samples_per_second: 0.7160
  eval_steps_per_second: 0.0900

Metric Comparison (Improvement/Regression):
  eval_loss: -0.0813
  eval_model_preparation_time: +0.0001
  eval_accuracy: +0.0000
  eval_precision: +0.0000
  eval_recall: +0.0000
  eval_f1: +0.0000
  eval_runtime: +7.0107
  eval_samples_per_second: -0.0190
  eval_steps_per_second: -0.0020
